# Tracer bullet — does the architecture hold?

A **tracer bullet**, not the product. It fires the thinnest end-to-end slice that
proves the chain works before v0.1.0 is built:

```
canonical records -> turn -> view -> teacher label -> majority vote
                  -> embedding -> logistic head -> session-disjoint eval -> metrics
```

Runs on a free Colab **CPU** runtime, with **no API key** and **no GPU**. Total
wall time is dominated by the first sentence-transformers model download (~90 MB).

**What it deliberately omits** (all v0.1.0 plan items): adapters, Parquet/DuckDB,
the real teacher, sampling-arm comparison, calibration, abstention, ONNX export,
the ship rule, and the `unmapped` discovery loop.

**One thing it cannot measure:** agreement across sampling arms. The teacher here
is a deterministic mock, so repeated passes are identical by construction and
$\alpha$ would be a meaningless 1.0. Arm selection needs a real teacher.


## 1. Environment

Colab ships `torch`/`transformers`/`pandas` at its own pinned versions. We install into **this** runtime so the result matches local/CI rather than Colab's defaults — silent version skew is exactly the class of difference the plan's `manifest.json` version recording exists to catch.

In [ ]:
# Colab-only bootstrap. Guarded so the notebook also runs locally unchanged.
import sys, subprocess, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers", "scikit-learn", "pandas", "pyyaml"],
                   check=True)
# The stack is pinned to <3.14: no torch wheel exists for 3.14, so an
# unpinned resolve installs a torch-free environment and every metric below
# would describe a different stack than the one CI pins. Fail loudly.
assert sys.version_info[:2] >= (3, 11) and sys.version_info[:2] < (3, 14), (
    f"python {sys.version.split()[0]} outside the supported range >=3.11,<3.14"
)
print("python", sys.version.split()[0])


## 2. Clone and install

The repo is public, so Colab can clone it directly. This is the same clone-and-run path any stranger would follow, which is the point — a tracer that only works on the author's machine proves nothing.

In [ ]:
REPO = "https://github.com/evanokeefe39/agent-turn-classifier.git"
if IN_COLAB and not os.path.isdir("/content/agent-turn-classifier"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/agent-turn-classifier"], check=True)

# Resolve the repo root ABSOLUTELY, then chdir once. Doing it the other way
# round (chdir to a relative path, then insert a relative "src") makes the
# import depend on the kernel's cwd, which is how this cell first failed under
# nbconvert with ModuleNotFoundError.
ROOT = "/content/agent-turn-classifier" if IN_COLAB else os.path.abspath(
    os.path.join(os.getcwd(), "..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
)
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("cwd:", os.getcwd())
print("src on path:", os.path.join(ROOT, "src"))
print(sorted(os.listdir(".")))


## 3. Validate the ontology first

Fail loudly before doing any work: a taxonomy that does not validate makes every downstream number meaningless. Note the anti-vacuity rule — an `excludes` reason must be ≥ 4 words, so a placeholder that validates while disambiguating nothing is rejected.

In [ ]:
from agent_turn_classifier import tracer as T

workflows, domains = T.load_ontology("examples/ontology.example.yaml")
errors = T.validate(workflows, domains)
print(f"{len(workflows)} workflows across {len(domains)} domains")
for w in workflows:
    print(f"  {w.id:10} {w.domain:18} {w.name}")
print()
print("validator errors:", errors if errors else "NONE")
assert not errors, "ontology must validate before anything downstream runs"


## 4. Records → turns → view

The view is the single text both teacher and student see. Redaction is applied *before* the hash is computed, so the hash attests the redacted payload — the egress seam's contract.

In [ ]:
turns = T.build_turns(T.read_canonical("examples/sessions.sample.jsonl"))
print(f"{len(turns)} turns from {len({t.session_id for t in turns})} sessions\n")

t0 = turns[0]
print(T.render(t0))
print("\nview sha256:", T.view_sha(T.render(t0)))


## 5. Teacher labels → majority vote

The mock reads canned labels keyed on the **rendered-view hash**, never the truth file. That matters: a mock that read back the answer would make the eval below vacuous.

In [ ]:
import json
from pathlib import Path
from collections import Counter

canned = {json.loads(l)["view_sha256"]: json.loads(l)["workflow"]
          for l in Path("examples/canned_labels.jsonl").read_text().splitlines() if l.strip()}
teacher = T.MockTeacher({T.render(t): canned.get(T.view_sha(T.render(t)), "unmapped") for t in turns})

majority_rows = {}
for t in turns:
    votes = [teacher.label(t, T.render(t)) for _ in range(3)]   # the plan's 3-pass arm
    majority_rows[t.turn_id] = T.majority(votes)

print("label distribution:", dict(Counter(r["workflow"] for r in majority_rows.values())))
print("a majority row:", json.dumps(next(iter(majority_rows.values())), indent=2))


## 6. Session-disjoint folds

Turns inside a session are correlated, so a random split leaks. The tracer **asserts** disjointness on the fitted folds rather than assuming it — the plan's P9 preflight does the same, on the grounds that intent is not evidence.

In [ ]:
views = {t.turn_id: T.render(t) for t in turns}
wf_domain = {w.id: w.domain for w in workflows}
path = lambda wf: f"{wf_domain[wf]}/{wf}" if wf in wf_domain else "unmapped"
labels = {k: path(v["workflow"]) for k, v in majority_rows.items()}

frame = T.build_frame(turns, views, labels)
print(f"{len(frame)} labelled turns, {frame['session_id'].nunique()} sessions, {frame['label'].nunique()} classes")
print()
print(frame.groupby("session_id")["label"].apply(list).to_string())


## 7. Train and evaluate

The student is an encoder plus a multinomial logistic head — the plan's default, chosen because labels-per-class here are tens-to-hundreds from weak labelling, not a few-shot regime. Primary metric is path macro-F1 over the flat `domain/workflow` label.

In [ ]:
import time, json
t0 = time.time()
metrics = T.run_tracer(folds=3)
metrics["wall_seconds"] = round(time.time() - t0, 1)
print(json.dumps(metrics, indent=2))


## 8. Read the result

Three numbers matter, and only one of them is the headline:

- **`student_path_macro_f1`** — the student against held-out truth.
- **`teacher_path_macro_f1`** — the *ceiling*. A student cannot beat its teacher.
- **`ceiling_gap`** — how much is lost in distillation. The plan ships a student
  only within `--max-gap-to-ceiling` (0.10) of the teacher, so this is the number
  that decides whether the approach is viable at all.

**Read this result honestly.** The corpus is 17 synthetic turns across 5 sessions
with 6 classes. At that size macro-F1 moves in steps of ~0.17 per class, so the
absolute value proves nothing about quality — it proves the *mechanism* runs end
to end, offline, on a free CPU runtime.

The question this tracer answers is: **does the chain hold together?** Not: is
the classifier good. Corpus size, teacher quality, and arm selection are v0.1.0
measurements against your real sessions.


In [ ]:
# The assertions that make this a tracer rather than a demo.
assert metrics["n_turns"] >= 20, "corpus below the density the design assumes"
assert metrics["n_sessions"] >= 3, "too few sessions for session-disjoint folds"
assert metrics["n_classes"] >= 3, "too few classes to be a meaningful path metric"
assert 0.0 <= metrics["student_path_macro_f1"] <= 1.0
assert metrics["ceiling_gap"] <= 0.10, (
    f"ceiling gap {metrics['ceiling_gap']} exceeds the ship rule's 0.10 — "
    "the corpus is likely below the density the design assumes"
)

# The label-space invariant. `none` (not work) and `unmapped` (work with no
# home) are different signals; folding one into the other silently corrupts
# both the coverage figure and the scored population. These counts must add up.
assert metrics["n_work"] == metrics["n_teacher_labelled"] - metrics["n_none"], (
    "work turns must exclude abstentions"
)
assert metrics["n_scored"] <= metrics["n_work"], (
    "scored turns cannot exceed work turns"
)
print("tracer OK: chain runs end to end, folds are session-disjoint, no key used")
